<a href="https://colab.research.google.com/github/yamamoto-irlab/sensing/blob/main/%E3%82%BB%E3%83%B3%E3%82%B7%E3%83%B3%E3%82%B0%E6%9C%80%E7%B5%82%E8%AA%B2%E9%A1%8C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q rosbags open3d

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.8/144.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 108.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 86.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 107.9 MB/s eta 0:00:00


In [ ]:
# ============================================================
# Import
# ============================================================

import numpy as np
import open3d as o3d

from pathlib import Path
from rosbags.highlevel import AnyReader
from rosbags.typesys import Stores, get_typestore
import plotly.graph_objects as go
from scipy.spatial import cKDTree

In [ ]:
# ============================================================
# 設定
# ============================================================

# ROS2 bag
bag_path = Path('/content/')
!ls -lh /content/ground

# 使用するトピック
topics = [
    '/livox/lidar',
    '/lidar_points'
]

# 何フレーム目から使うか
start_index = 10

# 使用するフレーム数
num_frames = 60

# 何フレームおきに使うか
# 1なら連続30フレーム
frame_step = 1

typestore = get_typestore(Stores.ROS2_HUMBLE)

ls: cannot access '/content/ground': No such file or directory


In [ ]:


# ============================================================
# LiDARごとの切り出し範囲
# ============================================================

filter_params = {

    # Livox
    '/livox/lidar': {
        'xmin': -2.0,
        'xmax':  2.0,
        'ymin': -2.0,
        'ymax':  2.0,
        'zmin': -0.50,
        'zmax': -0.15
    },

    # Hesai
    '/lidar_points': {
        'xmin': -5.0,
        'xmax':  5.0,
        'ymin': -5.0,
        'ymax':  5.0,
        'zmin': -0.70,
        'zmax': -0.20
    }
}

# ============================================================
# 可視化用
# ============================================================

visualization_clouds = {}

# ============================================================
# PointCloud2 → Nx3
# ============================================================

def pointcloud2_to_xyz(msg):

    dtype = np.dtype({
        'names': ['x', 'y', 'z'],
        'formats': [
            np.float32,
            np.float32,
            np.float32
        ],
        'offsets': [0, 4, 8],
        'itemsize': msg.point_step
    })

    points = np.frombuffer(
        msg.data,
        dtype=dtype
    )

    xyz = np.stack([
        points['x'],
        points['y'],
        points['z']
    ], axis=-1)

    # NaN・Inf除去
    xyz = xyz[
        np.isfinite(xyz).all(axis=1)
    ]

    return xyz


# ============================================================
# Livoxの取付姿勢補正
#
# URDF
#
# origin rpy="
# 1.5707963267948966
# 0
# 1.5707963267948966
# "
#
# Roll  = 90 deg
# Pitch = 0 deg
# Yaw   = 90 deg
# ============================================================

def rotate_livox(points):

    roll  = np.deg2rad(90.0)
    pitch = np.deg2rad(0.0)
    yaw   = np.deg2rad(90.0)

    # X軸回転
    Rx = np.array([
        [1, 0, 0],
        [0, np.cos(roll), -np.sin(roll)],
        [0, np.sin(roll),  np.cos(roll)]
    ])

    # Y軸回転
    Ry = np.array([
        [ np.cos(pitch), 0, np.sin(pitch)],
        [0,              1, 0],
        [-np.sin(pitch), 0, np.cos(pitch)]
    ])

    # Z軸回転
    Rz = np.array([
        [np.cos(yaw), -np.sin(yaw), 0],
        [np.sin(yaw),  np.cos(yaw), 0],
        [0,             0,           1]
    ])

    # URDFのRPY
    R = Rz @ Ry @ Rx

    return (R @ points.T).T


# ============================================================
# 点群切り出し
# ============================================================

def filter_pointcloud(
    points,
    xmin,
    xmax,
    ymin,
    ymax,
    zmin,
    zmax
):

    mask = (
        (points[:, 0] >= xmin) &
        (points[:, 0] <= xmax) &

        (points[:, 1] >= ymin) &
        (points[:, 1] <= ymax) &

        (points[:, 2] >= zmin) &
        (points[:, 2] <= zmax)
    )

    return points[mask]


# ============================================================
# RANSAC
#
# 1フレームについて
# Height・Roll・Pitchを計算
# ============================================================

def calculate_ground_pose(points):

    # 点が少なすぎる場合は使用しない
    if len(points) < 100:
        return None

    # Open3D形式へ変換
    pcd = o3d.geometry.PointCloud()

    pcd.points = (
        o3d.utility.Vector3dVector(points)
    )

    # --------------------------------------------------------
    # RANSACで地面平面を抽出
    # --------------------------------------------------------

    plane_model, inliers = pcd.segment_plane(
        distance_threshold=0.01,
        ransac_n=3,
        num_iterations=1000
    )

    # 地面点数
    num_ground = len(inliers)

    # 地面割合
    ground_ratio = (
        num_ground /
        len(points) *
        100
    )

    # --------------------------------------------------------
    # 平面
    #
    # ax + by + cz + d = 0
    # --------------------------------------------------------

    a, b, c, d = plane_model

    # --------------------------------------------------------
    # 法線を正規化
    # --------------------------------------------------------

    norm = np.sqrt(
        a**2 +
        b**2 +
        c**2
    )

    a /= norm
    b /= norm
    c /= norm
    d /= norm

    # --------------------------------------------------------
    # 法線を上向きに統一
    # --------------------------------------------------------

    if c < 0:

        a = -a
        b = -b
        c = -c
        d = -d

    # --------------------------------------------------------
    # 地面として妥当か確認
    #
    # c = 1 に近いほど水平面
    #
    # c < 0.90なら
    # 地面以外を拾った可能性があるので除外
    # --------------------------------------------------------

    if c < 0.90:
        return None

    # --------------------------------------------------------
    # 高さ
    # --------------------------------------------------------

    height = abs(d)

    # --------------------------------------------------------
    # Roll
    # --------------------------------------------------------

    roll = np.degrees(
        np.arctan2(
            b,
            c
        )
    )

    # --------------------------------------------------------
    # Pitch
    # --------------------------------------------------------

    pitch = np.degrees(
        np.arctan2(
            -a,
            np.sqrt(
                b**2 +
                c**2
            )
        )
    )

    ground_cloud = np.asarray(
    pcd.select_by_index(inliers).points
    )

    return {
        'height': height,
        'roll': roll,
        'pitch': pitch,
        'ground_points': num_ground,
        'ground_ratio': ground_ratio,
        'ground_cloud': ground_cloud
    }


# ============================================================
# 指定トピックを複数フレーム処理
# ============================================================

def process_topic(
    reader,
    topic,
    start_index,
    num_frames,
    frame_step
):

    connections = [
        c for c in reader.connections
        if c.topic == topic
    ]

    if len(connections) == 0:
        raise RuntimeError(
            f"Topic not found: {topic}"
        )

    # 使用したいフレーム番号
    target_indices = set(
        start_index +
        i * frame_step
        for i in range(num_frames)
    )

    results = []

    frame_id = None

    for msg_idx, (
        connection,
        timestamp,
        rawdata
    ) in enumerate(
        reader.messages(
            connections=connections
        )
    ):

        # 必要なフレームだけ使う
        if msg_idx not in target_indices:
            continue

        # Deserialize
        msg = reader.deserialize(
            rawdata,
            connection.msgtype
        )

        if frame_id is None:
            frame_id = msg.header.frame_id

        # PointCloud2 → numpy
        pc = pointcloud2_to_xyz(msg)

        # ----------------------------------------------------
        # Livoxだけ既知の取付角を補正
        # ----------------------------------------------------

        if topic == '/livox/lidar':
            pc = rotate_livox(pc)

        # ----------------------------------------------------
        # LiDARごとの範囲で切り出し
        # ----------------------------------------------------

        params = filter_params[topic]

        pc_f = filter_pointcloud(
            pc,
            xmin=params['xmin'],
            xmax=params['xmax'],
            ymin=params['ymin'],
            ymax=params['ymax'],
            zmin=params['zmin'],
            zmax=params['zmax']
        )

        # ----------------------------------------------------
        # RANSAC
        # ----------------------------------------------------

        result = calculate_ground_pose(
            pc_f
        )

        # RANSAC失敗・地面でない場合
        if result is None:

            print(
                f"Frame {msg_idx:3d}: "
                f"除外"
            )

            continue

        result['frame'] = msg_idx
        result['filtered_points'] = len(pc_f)

        if topic not in visualization_clouds:
          visualization_clouds[topic] = {
              'pc_f': pc_f.copy(),
              'ground': result['ground_cloud'].copy()
              }

        results.append(result)

        # 必要フレームを全部取得したら終了
        if len([
            i for i in target_indices
            if i <= msg_idx
        ]) >= num_frames:
            break

    return frame_id, results


# ============================================================
# 統計結果表示
# ============================================================

def print_summary(
    topic,
    frame_id,
    results
):

    print()
    print("=" * 65)
    print("RESULT")
    print("=" * 65)

    print("Topic    :", topic)
    print("Frame ID :", frame_id)

    print(
        "有効フレーム数:",
        len(results)
    )

    if len(results) == 0:

        print(
            "有効な地面平面がありません"
        )

        return

    # --------------------------------------------------------
    # numpy array
    # --------------------------------------------------------

    heights = np.array([
        r['height']
        for r in results
    ])

    rolls = np.array([
        r['roll']
        for r in results
    ])

    pitches = np.array([
        r['pitch']
        for r in results
    ])

    ground_ratios = np.array([
        r['ground_ratio']
        for r in results
    ])

    # --------------------------------------------------------
    # 中央値
    # --------------------------------------------------------

    height_median = np.median(
        heights
    )

    roll_median = np.median(
        rolls
    )

    pitch_median = np.median(
        pitches
    )

    # --------------------------------------------------------
    # 平均
    # --------------------------------------------------------

    height_mean = np.mean(
        heights
    )

    roll_mean = np.mean(
        rolls
    )

    pitch_mean = np.mean(
        pitches
    )

    # --------------------------------------------------------
    # 標準偏差
    # --------------------------------------------------------

    height_std = np.std(
        heights
    )

    roll_std = np.std(
        rolls
    )

    pitch_std = np.std(
        pitches
    )

    # ========================================================
    # 結果表示
    # ========================================================

    print()
    print(
        "---------- 中央値 ----------"
    )

    print(
        f"Height : "
        f"{height_median:.4f} m"
    )

    print(
        f"Roll   : "
        f"{roll_median:.4f} deg"
    )

    print(
        f"Pitch  : "
        f"{pitch_median:.4f} deg"
    )

    print()
    print(
        "---------- 平均 ----------"
    )

    print(
        f"Height : "
        f"{height_mean:.4f} m"
    )

    print(
        f"Roll   : "
        f"{roll_mean:.4f} deg"
    )

    print(
        f"Pitch  : "
        f"{pitch_mean:.4f} deg"
    )

    print()
    print(
        "---------- 標準偏差 ----------"
    )

    print(
        f"Height : "
        f"{height_std:.4f} m"
    )

    print(
        f"Roll   : "
        f"{roll_std:.4f} deg"
    )

    print(
        f"Pitch  : "
        f"{pitch_std:.4f} deg"
    )

    print()

    print(
        f"Height = "
        f"{height_median:.4f} m"
    )

    print(
        f"Roll   = "
        f"{roll_median:.4f} deg"
    )

    print(
        f"Pitch  = "
        f"{pitch_median:.4f} deg"
    )

# ============================================================
# 実行
# ============================================================

with AnyReader(
    [bag_path],
    default_typestore=typestore
) as reader:

    for topic in topics:

        print()
        print()

        frame_id, results = process_topic(
            reader,
            topic,
            start_index,
            num_frames,
            frame_step
        )

        print_summary(
            topic,
            frame_id,
            results
        )




RESULT
Topic    : /livox/lidar
Frame ID : livox_frame
有効フレーム数: 45

---------- 中央値 ----------
Height : 0.3259 m
Roll   : 0.9564 deg
Pitch  : -1.0375 deg

---------- 平均 ----------
Height : 0.3259 m
Roll   : 0.9604 deg
Pitch  : -1.0554 deg

---------- 標準偏差 ----------
Height : 0.0007 m
Roll   : 0.0179 deg
Pitch  : 0.0706 deg

Height = 0.3259 m
Roll   = 0.9564 deg
Pitch  = -1.0375 deg



RESULT
Topic    : /lidar_points
Frame ID : hesai_lidar
有効フレーム数: 45

---------- 中央値 ----------
Height : 0.4633 m
Roll   : -0.4607 deg
Pitch  : 0.1379 deg

---------- 平均 ----------
Height : 0.4633 m
Roll   : -0.4606 deg
Pitch  : 0.1368 deg

---------- 標準偏差 ----------
Height : 0.0002 m
Roll   : 0.0066 deg
Pitch  : 0.0058 deg

Height = 0.4633 m
Roll   = -0.4607 deg
Pitch  = 0.1379 deg


In [ ]:
def show_cloud(points, title):

    fig = go.Figure()

    fig.add_trace(
        go.Scatter3d(
            x=points[:, 0],
            y=points[:, 1],
            z=points[:, 2],
            mode='markers',
            marker=dict(
                size=2,
                color=points[:, 2],
                colorscale='Viridis'
            )
        )
    )

    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title='X [m]',
            yaxis_title='Y [m]',
            zaxis_title='Z [m]',
            aspectmode='data'
        )
    )

    fig.show()

In [ ]:
#点群の可視化
for topic in ['/livox/lidar', '/lidar_points']:

    pc_f = visualization_clouds[topic]['pc_f']
    ground = visualization_clouds[topic]['ground']

    show_cloud(
        pc_f,
        f'{topic} : 切り出した点群'
    )

    show_cloud(
        ground,
        f'{topic} : RANSACで抽出した地面'
    )

**以下ICPによるLiDAR間の測定のためのコード**

In [ ]:
# ============================================================
# bag
# ============================================================

bag_path = Path('/content/icp')

livox_topic = '/livox/lidar'
hesai_topic = '/lidar_points'

reference_index = 10

In [ ]:
# ============================================================
# 既知の取り付け回転
# ============================================================

# Livox
livox_mount_roll  = 90.0
livox_mount_pitch = 0.0
livox_mount_yaw   = 90.0

# Hesai
hesai_mount_roll  = 0.0
hesai_mount_pitch = 0.0
hesai_mount_yaw   = 90.0


# ============================================================
# ICP初期値
# ============================================================

init_x = 0.09
print(init_x)
init_y = 0.00
# 地面推定から得た高さ差
fixed_z = -0.135

# ============================================================
# 重複領域
#
# Hesai座標系上で指定
# ============================================================

overlap_roi = {

    'xmin': -0.5,
    'xmax':  2.5,

    'ymin': -1.8,
    'ymax':  2.5,

    'zmin': -0.35,
    'zmax':  1.20
}


# ============================================================
# ICP設定
# ============================================================

voxel_size = 0.05

# 2段階
coarse_distance = 0.25
fine_distance   = 0.10

max_iterations = 40


# ============================================================
# PointCloud取得
# ============================================================

def get_cloud_by_index(reader, topic, index=10):

    conns = [
        c for c in reader.connections
        if c.topic == topic
    ]

    if len(conns) == 0:
        raise RuntimeError(
            f"Topic not found: {topic}"
        )

    for i, (conn, timestamp, rawdata) in enumerate(
        reader.messages(connections=conns)
    ):

        if i == index:

            msg = reader.deserialize(
                rawdata,
                conn.msgtype
            )

            cloud = pointcloud2_to_xyz(msg)

            return timestamp, cloud

    raise RuntimeError(
        f"{topic}: index={index} が見つかりません"
    )


# ============================================================
# 指定時刻に最も近い点群取得
# ============================================================

def get_nearest_cloud(
    reader,
    topic,
    target_timestamp
):

    conns = [
        c for c in reader.connections
        if c.topic == topic
    ]

    if len(conns) == 0:
        raise RuntimeError(
            f"Topic not found: {topic}"
        )

    best_dt = None
    best_timestamp = None
    best_cloud = None

    for conn, timestamp, rawdata in reader.messages(
        connections=conns
    ):

        dt = abs(
            timestamp - target_timestamp
        )

        if best_dt is None or dt < best_dt:

            msg = reader.deserialize(
                rawdata,
                conn.msgtype
            )

            best_dt = dt
            best_timestamp = timestamp
            best_cloud = pointcloud2_to_xyz(msg)

    return best_timestamp, best_cloud


# ============================================================
# 回転
# ============================================================

def rotate_points(
    points,
    roll_deg,
    pitch_deg,
    yaw_deg
):

    roll = np.deg2rad(
        roll_deg
    )

    pitch = np.deg2rad(
        pitch_deg
    )

    yaw = np.deg2rad(
        yaw_deg
    )

    Rx = np.array([
        [1, 0, 0],
        [
            0,
            np.cos(roll),
            -np.sin(roll)
        ],
        [
            0,
            np.sin(roll),
            np.cos(roll)
        ]
    ])

    Ry = np.array([
        [
            np.cos(pitch),
            0,
            np.sin(pitch)
        ],
        [0, 1, 0],
        [
            -np.sin(pitch),
            0,
            np.cos(pitch)
        ]
    ])

    Rz = np.array([
        [
            np.cos(yaw),
            -np.sin(yaw),
            0
        ],
        [
            np.sin(yaw),
            np.cos(yaw),
            0
        ],
        [0, 0, 1]
    ])

    R = Rz @ Ry @ Rx

    return (
        R @ points.T
    ).T


# ============================================================
# 変換行列作成
# ============================================================

def make_transform(
    x,
    y,
    z,
    roll_deg,
    pitch_deg,
    yaw_deg
):

    roll = np.deg2rad(
        roll_deg
    )

    pitch = np.deg2rad(
        pitch_deg
    )

    yaw = np.deg2rad(
        yaw_deg
    )

    Rx = np.array([
        [1, 0, 0],
        [
            0,
            np.cos(roll),
            -np.sin(roll)
        ],
        [
            0,
            np.sin(roll),
            np.cos(roll)
        ]
    ])

    Ry = np.array([
        [
            np.cos(pitch),
            0,
            np.sin(pitch)
        ],
        [0, 1, 0],
        [
            -np.sin(pitch),
            0,
            np.cos(pitch)
        ]
    ])

    Rz = np.array([
        [
            np.cos(yaw),
            -np.sin(yaw),
            0
        ],
        [
            np.sin(yaw),
            np.cos(yaw),
            0
        ],
        [0, 0, 1]
    ])

    T = np.eye(4)

    T[:3, :3] = (
        Rz @ Ry @ Rx
    )

    T[:3, 3] = [
        x,
        y,
        z
    ]

    return T


# ============================================================
# 変換適用
# ============================================================

def apply_transform(
    points,
    T
):

    R = T[:3, :3]
    t = T[:3, 3]

    return (
        R @ points.T
    ).T + t


# ============================================================
# ROI切り出し
# ============================================================

def crop_roi(
    points,
    roi
):

    mask = (

        (points[:, 0] >= roi['xmin']) &
        (points[:, 0] <= roi['xmax']) &

        (points[:, 1] >= roi['ymin']) &
        (points[:, 1] <= roi['ymax']) &

        (points[:, 2] >= roi['zmin']) &
        (points[:, 2] <= roi['zmax'])
    )

    return points[mask]


# ============================================================
# 2D剛体変換推定
#
# x, y, yaw のみ
# ============================================================

def estimate_xy_yaw(
    source_xy,
    target_xy
):

    source_center = np.mean(
        source_xy,
        axis=0
    )

    target_center = np.mean(
        target_xy,
        axis=0
    )

    source_zero = (
        source_xy -
        source_center
    )

    target_zero = (
        target_xy -
        target_center
    )

    H = (
        source_zero.T @
        target_zero
    )

    U, S, Vt = np.linalg.svd(
        H
    )

    R2 = (
        Vt.T @ U.T
    )

    # 反転防止
    if np.linalg.det(R2) < 0:

        Vt[-1, :] *= -1

        R2 = (
            Vt.T @ U.T
        )

    t2 = (
        target_center -
        R2 @ source_center
    )

    return R2, t2


# ============================================================
# x,y,yaw 限定ICP
# ============================================================

def constrained_icp_xyyaw(
    source_points,
    target_points,
    T_init,
    max_distance,
    roi,
    max_iterations=40
):

    T = T_init.copy()

    # Hesai側は固定なので先にROI切り出し
    target_roi = crop_roi(
        target_points,
        roi
    )

    tree = cKDTree(
        target_roi
    )

    for iteration in range(
        max_iterations
    ):

        # 現在の変換
        transformed = apply_transform(
            source_points,
            T
        )

        # LivoxもHesai座標系上でROI切り出し
        source_roi = crop_roi(
            transformed,
            roi
        )

        if len(source_roi) < 20:
            print(
                "対応するLivox点が少なすぎます"
            )
            break

        # 最近傍
        distances, indices = tree.query(
            source_roi,
            k=1
        )

        valid = (
            distances <
            max_distance
        )

        if np.sum(valid) < 20:

            print(
                f"Iteration {iteration}: "
                "対応点不足"
            )

            break

        src_corr = (
            source_roi[valid]
        )

        tgt_corr = (
            target_roi[
                indices[valid]
            ]
        )

        dist_corr = (
            distances[valid]
        )

        # ----------------------------------------------------
        # 外れ値をさらに除去
        #
        # 距離の近い80%だけ使用
        # ----------------------------------------------------

        threshold = np.quantile(
            dist_corr,
            0.80
        )

        robust = (
            dist_corr <= threshold
        )

        src_corr = (
            src_corr[robust]
        )

        tgt_corr = (
            tgt_corr[robust]
        )

        dist_corr = (
            dist_corr[robust]
        )

        # ----------------------------------------------------
        # XYだけ使って変換推定
        # ----------------------------------------------------

        R2, t2 = estimate_xy_yaw(

            src_corr[:, :2],

            tgt_corr[:, :2]
        )

        # 追加変換
        Delta = np.eye(4)

        Delta[0:2, 0:2] = R2
        Delta[0:2, 3] = t2

        # 左から掛ける
        T_new = (
            Delta @ T
        )

        # ----------------------------------------------------
        # 収束判定
        # ----------------------------------------------------

        dx = (
            T_new[0, 3] -
            T[0, 3]
        )

        dy = (
            T_new[1, 3] -
            T[1, 3]
        )

        delta_translation = np.sqrt(
            dx**2 +
            dy**2
        )

        delta_yaw = np.rad2deg(
            np.arctan2(
                R2[1, 0],
                R2[0, 0]
            )
        )

        T = T_new

        rmse = np.sqrt(
            np.mean(
                dist_corr**2
            )
        )

        print(
            f"Iter {iteration:2d}: "
            f"corr={len(src_corr):5d}, "
            f"RMSE={rmse:.4f} m, "
            f"dxy={delta_translation:.5f} m, "
            f"dyaw={delta_yaw:+.4f} deg"
        )

        # 十分小さくなったら終了
        if (
            delta_translation < 0.0001
            and
            abs(delta_yaw) < 0.01
        ):
            break

    return T


# ============================================================
# T → xyz RPY
# ============================================================

def transform_to_xyzrpy(
    T
):

    x = T[0, 3]
    y = T[1, 3]
    z = T[2, 3]

    R = T[:3, :3]

    pitch = np.arcsin(
        np.clip(
            -R[2, 0],
            -1.0,
            1.0
        )
    )

    roll = np.arctan2(
        R[2, 1],
        R[2, 2]
    )

    yaw = np.arctan2(
        R[1, 0],
        R[0, 0]
    )

    return (
        x,
        y,
        z,
        np.rad2deg(roll),
        np.rad2deg(pitch),
        np.rad2deg(yaw)
    )


# ============================================================
# 3D表示
# ============================================================

def show_overlay(
    hesai_points,
    livox_points,
    title
):

    fig = go.Figure()

    fig.add_trace(
        go.Scatter3d(
            x=hesai_points[:, 0],
            y=hesai_points[:, 1],
            z=hesai_points[:, 2],
            mode='markers',
            marker=dict(
                size=1,
                color='blue'
            ),
            name='Hesai'
        )
    )

    fig.add_trace(
        go.Scatter3d(
            x=livox_points[:, 0],
            y=livox_points[:, 1],
            z=livox_points[:, 2],
            mode='markers',
            marker=dict(
                size=1,
                color='red'
            ),
            name='Livox'
        )
    )

    fig.update_layout(
        title=title,

        scene=dict(
            xaxis_title='X [m]',
            yaxis_title='Y [m]',
            zaxis_title='Z [m]',
            aspectmode='data'
        )
    )

    fig.show()

# ============================================================
# データ取得
# ============================================================

with AnyReader(
    [bag_path],
    default_typestore=typestore
) as reader:

    livox_time, livox = get_cloud_by_index(
        reader,
        livox_topic,
        reference_index
    )

    hesai_time, hesai = get_nearest_cloud(
        reader,
        hesai_topic,
        livox_time
    )


print()


# ============================================================
# 取付角補正
# ============================================================

livox = rotate_points(
    livox,
    livox_mount_roll,
    livox_mount_pitch,
    livox_mount_yaw
)

hesai = rotate_points(
    hesai,
    hesai_mount_roll,
    hesai_mount_pitch,
    hesai_mount_yaw
)


# ============================================================
# Voxel downsample
# ============================================================

livox_pcd = o3d.geometry.PointCloud()

livox_pcd.points = (
    o3d.utility.Vector3dVector(
        livox
    )
)

hesai_pcd = o3d.geometry.PointCloud()

hesai_pcd.points = (
    o3d.utility.Vector3dVector(
        hesai
    )
)

livox_pcd = (
    livox_pcd.voxel_down_sample(
        voxel_size
    )
)

hesai_pcd = (
    hesai_pcd.voxel_down_sample(
        voxel_size
    )
)

livox_points = np.asarray(
    livox_pcd.points
)

hesai_points = np.asarray(
    hesai_pcd.points
)


# ============================================================
# 初期変換
# ============================================================

T_init = make_transform(

    init_x,
    init_y,
    fixed_z,

    fixed_roll,
    fixed_pitch,

    init_yaw
)


# ============================================================
# ICP前
# ============================================================

livox_before = apply_transform(
    livox_points,
    T_init
)

livox_before_roi = crop_roi(
    livox_before,
    overlap_roi
)

hesai_roi = crop_roi(
    hesai_points,
    overlap_roi
)

print()
print("===== 重複領域 =====")

print(
    f"Livox : {len(livox_before_roi)} points"
)

print(
    f"Hesai : {len(hesai_roi)} points"
)


show_overlay(
    hesai_roi,
    livox_before_roi,
    'ICP前：重複領域'
)


# ============================================================
# Coarse ICP
# ============================================================

print()
print("===== Coarse ICP =====")

T_coarse = constrained_icp_xyyaw(

    livox_points,
    hesai_points,

    T_init,

    coarse_distance,

    overlap_roi,

    max_iterations
)


# ============================================================
# Fine ICP
# ============================================================

print()
print("===== Fine ICP =====")

T_final = constrained_icp_xyyaw(

    livox_points,
    hesai_points,

    T_coarse,

    fine_distance,

    overlap_roi,

    max_iterations
)


# ============================================================
# 最終結果
# ============================================================

x, y, z, roll, pitch, yaw = (
    transform_to_xyzrpy(
        T_final
    )
)

print()
print(
    "===== Final Livox → Hesai ====="
)

print(
    f"x     = {x:+.4f} m"
)

print(
    f"y     = {y:+.4f} m"
)

print(
    f"z     = {z:+.4f} m"
)

print(
    f"roll  = {roll:+.3f} deg"
)

print(
    f"pitch = {pitch:+.3f} deg"
)

print(
    f"yaw   = {yaw:+.3f} deg"
)


# ============================================================
# ICP後
# ============================================================

livox_after = apply_transform(
    livox_points,
    T_final
)

livox_after_roi = crop_roi(
    livox_after,
    overlap_roi
)


show_overlay(
    hesai_roi,
    livox_after_roi,
    'ICP後：重複領域'
)


# ============================================================
# ICP後 XY
# ============================================================

show_2d_overlay(
    hesai_roi,
    livox_after_roi,
    'X',
    'Y',
    'ICP後：XY平面'
)


# ============================================================
# ICP後 XZ
# ============================================================

show_2d_overlay(
    hesai_roi,
    livox_after_roi,
    'X',
    'Z',
    'ICP後：XZ平面'
)


# ============================================================
# ICP後 YZ
# ============================================================

show_2d_overlay(
    hesai_roi,
    livox_after_roi,
    'Y',
    'Z',
    'ICP後：YZ平面'
)

0.09


===== 重複領域 =====
Livox : 102 points
Hesai : 191 points



===== Coarse ICP =====
Iter  0: corr=   80, RMSE=0.0497 m, dxy=0.04850 m, dyaw=+1.3471 deg
Iter  1: corr=   80, RMSE=0.0444 m, dxy=0.00825 m, dyaw=+0.0445 deg
Iter  2: corr=   82, RMSE=0.0427 m, dxy=0.04646 m, dyaw=-0.8874 deg
Iter  3: corr=   81, RMSE=0.0425 m, dxy=0.01750 m, dyaw=-0.2729 deg
Iter  4: corr=   81, RMSE=0.0419 m, dxy=0.00660 m, dyaw=-0.0736 deg
Iter  5: corr=   81, RMSE=0.0417 m, dxy=0.00633 m, dyaw=-0.1027 deg
Iter  6: corr=   81, RMSE=0.0416 m, dxy=0.00324 m, dyaw=-0.0534 deg
Iter  7: corr=   81, RMSE=0.0416 m, dxy=0.00193 m, dyaw=-0.0274 deg
Iter  8: corr=   81, RMSE=0.0416 m, dxy=0.00078 m, dyaw=-0.0102 deg
Iter  9: corr=   81, RMSE=0.0416 m, dxy=0.00000 m, dyaw=+0.0000 deg

===== Fine ICP =====
Iter  0: corr=   70, RMSE=0.0358 m, dxy=0.00805 m, dyaw=-0.1533 deg
Iter  1: corr=   70, RMSE=0.0364 m, dxy=0.00631 m, dyaw=-0.1027 deg
Iter  2: corr=   69, RMSE=0.0359 m, dxy=0.00716 m, dyaw=-0.1196 deg
Iter  3: corr=   70, RMSE=0.0356 m, dxy=0.00249 m, dyaw=-0.0539 deg
It